# Notebook 3 — Interactive Hazard Maps
## GeoINquire Workshop · Messina, April 2026

**Purpose:** Download spatial hazard grids from the EFEHR/SHARE API and produce geographic maps of seismic hazard over Italy and the Mediterranean. Unlike Notebooks 1 and 2 — which work with a single point — this notebook queries the `/map` endpoint which returns a pre-computed spatial grid.

### What you will learn
- How to download and parse a spatial hazard map from the EFEHR API
- How to plot and interpret ground-motion maps with a geographic backdrop
- How spatial hazard patterns relate to tectonic structure
- How hazard changes with return period across a region
- How to compare two model generations spatially (ESHM13 vs. ESHM20)
- How to create interactive web maps with site-specific information

### Workflow
```
Notebook 1           →  hazard_config.yaml  →  Notebook 3
(discovery, Messina)     (models + PoEs)        (spatial maps)
```

**Prerequisite:** `hazard_config.yaml` from Notebook 1 must exist in the same folder.

---
### The `/map` endpoint

The SHARE API exposes a `/map` endpoint alongside `/curve` and `/spectra`. While `/curve` returns hazard at a single site and `/spectra` returns the response spectrum at a single site, `/map` returns a spatial grid of hazard values over the full model domain.

```
/map?id={model_id}&imt={imt}                           → available PoEs for mapping
/map?id={model_id}&imt={imt}&poe={P}&timespanpoe={T}
    &soiltype={soil}&aggregationtype={agg}&aggregationlevel={level}
                                                        → spatial grid of IML values
```

Each grid node contains a longitude, latitude, and IML value. The grid spacing is defined by the model (~0.1° for ESHM20 over Europe). A full-domain request returns tens of thousands of nodes — the response is a large XML file.

---
## 1. Install dependencies (run once)

In [ ]:
import subprocess, sys

# Core requirements — always needed
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pyyaml', 'requests', 'matplotlib', 'numpy', 'scipy'])

# Optional: cartopy for proper coastlines and map projections
# If this fails, the notebook falls back to a pure-matplotlib map
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cartopy'])
    print('cartopy installed.')
except Exception:
    print('cartopy not available — will use fallback matplotlib map.')

# Optional: folium for interactive HTML maps
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'folium'])
    print('folium installed.')
except Exception:
    print('folium not available — Exercise E will be skipped.')

print('\nAll required dependencies ready.')

---
## 2. Imports and environment detection

In [ ]:
import requests
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import BoundaryNorm, LogNorm
import numpy as np
from scipy.interpolate import griddata
import io, yaml, sys, math

# Optional imports — degrade gracefully if not available
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
    print('cartopy available: geographic projections and coastlines enabled.')
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not available: using simple matplotlib maps.')

try:
    import folium
    HAS_FOLIUM = True
    print('folium available: interactive HTML maps enabled.')
except ImportError:
    HAS_FOLIUM = False
    print('folium not available: Exercise E will be skipped.')

BASE_URL    = 'http://appsrvr.share-eu.org:8080/share'
CONFIG_FILE = 'hazard_config.yaml'
MAP_POES    = {}      # populated by the probe cell; guards resolve_poe() NameError
MODELS      = {}      # populated by Cell 5 (load_config); guards exercise cells

---
## 3. API helpers and map data fetcher

In [ ]:

# =============================================================================
# SHARE /map API — how it actually works (discovered through debugging)
#
# The /map endpoint is a 3-level cascade, not a single data endpoint:
#
#  Level 1: /map?id=M&imt=I
#            → <exceedances> listing available PoEs
#
#  Level 2: /map?id=M&imt=I&hmapexceedprob=P&hmapexceedyears=Y
#            → <fractiles> listing available aggregations
#
#  Level 3: /map?id=M&imt=I&hmapexceedprob=P&hmapexceedyears=Y
#                          &aggregationtype=T&aggregationlevel=L
#            → <hmapid>, <hmapwms>, <hazardmaplocation>
#               (a reference to the pre-computed map, NOT raw grid data)
#
#  Level 4: GET <hazardmaplocation>
#            → the actual map data (CSV, ASCII grid, or similar)
# =============================================================================

def get_xml_root(url, debug=False):
    """Fetch URL, parse XML, strip namespace prefixes. Returns (root, raw_text)."""
    try:
        if debug:
            print(f'  GET {url}')
        r = requests.get(url, timeout=120)
        if r.status_code != 200:
            if debug:
                print(f'  HTTP {r.status_code}: {r.text[:200]}')
            return None, r.text if r.text else None
        it = ET.iterparse(io.StringIO(r.text))
        for _, el in it:
            _, _, el.tag = el.tag.rpartition('}')
        return it.root, r.text
    except Exception as e:
        if debug:
            print(f'  Error: {e}')
        return None, None


def probe_map_poes(model_id, imt):
    """Level 1: return list of {prob, years} dicts available for this model/IMT."""
    url = f'{BASE_URL}/map?id={model_id}&imt={imt}'
    root, _ = get_xml_root(url)
    if root is None:
        return []
    poes, seen = [], set()
    for exc in root.iter():
        if 'exceedance' not in exc.tag.lower():
            continue
        p = exc.find('hmapexceedprob')
        y = exc.find('hmapexceedyears')
        if p is not None and y is not None and p.text and y.text:
            key = (p.text.strip(), y.text.strip())
            if key not in seen:
                seen.add(key)
                poes.append({'prob': key[0], 'years': key[1]})
    return poes


def probe_map_aggregations(model_id, imt, poe_prob, poe_years):
    """Level 2: return list of {type, level} dicts for this model/IMT/PoE."""
    url = (f'{BASE_URL}/map?id={model_id}&imt={imt}'
           f'&hmapexceedprob={poe_prob}&hmapexceedyears={poe_years}')
    root, _ = get_xml_root(url)
    if root is None:
        return []
    aggs, seen = [], set()
    for f in root.iter():
        if 'fractile' not in f.tag.lower():
            continue
        t = f.find('aggregationtype')
        l = f.find('aggregationlevel')
        if t is not None and l is not None and t.text and l.text:
            key = (t.text.strip(), l.text.strip())
            if key not in seen:
                seen.add(key)
                aggs.append({'type': key[0], 'level': key[1]})
    return aggs


def get_map_reference(model_id, imt, poe_prob, poe_years,
                       agg_type, agg_level, soil, debug=False):
    """
    Level 3: fetch the map reference.
    Returns dict with keys: hmapid, hmapwms, hazardmaplocation (all strings).
    Returns None on failure.
    """
    url = (f'{BASE_URL}/map?id={model_id}&imt={imt}'
           f'&hmapexceedprob={poe_prob}&hmapexceedyears={poe_years}'
           f'&soiltype={soil}'
           f'&aggregationtype={agg_type}&aggregationlevel={agg_level}')
    if debug:
        print(f'  Level 3 URL: {url}')
    root, raw = get_xml_root(url, debug=debug)
    if root is None:
        return None

    ref = {}
    for elem in root.iter():
        tag = elem.tag.lower()
        txt = elem.text.strip() if elem.text and elem.text.strip() else None
        if not txt:
            continue
        # Known keys
        if 'hmapid'            in tag: ref['hmapid']            = txt
        if 'hmapwms'           in tag: ref['hmapwms']           = txt
        if 'hazardmaplocation' in tag: ref['hazardmaplocation'] = txt
        # Catch-all: any tag whose text looks like a URL → save as hazardmaplocation
        if 'hazardmaplocation' not in ref:
            if txt.startswith('http') and ('map' in tag or 'location' in tag
                                            or 'url' in tag or 'data' in tag):
                ref['hazardmaplocation'] = txt

    if debug:
        print(f'  All elements in Level 3 response:')
        for elem in root.iter():
            if elem.text and elem.text.strip():
                print(f'    <{elem.tag}> {elem.text.strip()[:120]}')
        print(f'  Parsed reference: {ref}')
    return ref if ref else None


def fetch_map_data_from_location(location_url, bbox=None, debug=False):
    """
    Level 4: download the actual map data from <hazardmaplocation>.
    Tries CSV, space-delimited, and XML parsing in order.
    Returns (lons, lats, values) as numpy arrays, or (None, None, None).
    """
    if debug:
        print(f'  Fetching data from: {location_url}')
    try:
        r = requests.get(location_url, timeout=120)
        if r.status_code != 200:
            if debug:
                print(f'  HTTP {r.status_code}')
            return None, None, None
        content = r.text
        if debug:
            print(f'  Response: {len(content)} chars, '
                  f'first 150: {content[:150]!r}')

        lons, lats, vals = [], [], []

        # ── Try 1: CSV  lon,lat,iml  or  lat,lon,iml ──────────────────
        lines = [l.strip() for l in content.strip().splitlines() if l.strip()]
        if lines and (',' in lines[0] or '\t' in lines[0]):
            sep = ',' if ',' in lines[0] else '\t'
            # Skip header lines
            data_lines = [l for l in lines if l.replace('.','').replace('-','')
                          .replace('E','').replace('e','').replace(sep,'')
                          .replace(' ','').lstrip('-').isdigit()
                          or _is_float_row(l, sep)]
            for line in data_lines:
                parts = [p.strip() for p in line.split(sep)]
                if len(parts) >= 3:
                    try:
                        # Try lon,lat,val first, then lat,lon,val
                        a, b, c = float(parts[0]), float(parts[1]), float(parts[2])
                        # Heuristic for European data: latitude is always larger
                        # than longitude for Mediterranean/European sites.
                        # (e.g. lat=38 > lon=15 for Italy; lat=48 > lon=2 for France)
                        # abs(larger) is latitude; abs(smaller) is longitude.
                        if abs(a) >= abs(b):  # a is probably latitude
                            lats.append(a); lons.append(b); vals.append(c)
                        else:                 # a is probably longitude
                            lons.append(a); lats.append(b); vals.append(c)
                    except ValueError:
                        pass
            if lons:
                if debug: print(f'  CSV parser: {len(lons)} nodes')
                return _apply_bbox(np.array(lons), np.array(lats),
                                   np.array(vals), bbox)

        # ── Try 2: Space-delimited rows ────────────────────────────────
        for line in lines:
            parts = line.split()
            if len(parts) >= 3:
                try:
                    a, b, c = float(parts[0]), float(parts[1]), float(parts[2])
                    # Same European heuristic: |lat| > |lon|
                    if abs(a) >= abs(b):
                        lats.append(a); lons.append(b)
                    else:
                        lons.append(a); lats.append(b)
                    vals.append(c)
                except ValueError:
                    pass
        if lons:
            if debug: print(f'  Space-delimited parser: {len(lons)} nodes')
            return _apply_bbox(np.array(lons), np.array(lats),
                               np.array(vals), bbox)

        # ── Try 3: XML node format ─────────────────────────────────────
        try:
            it = ET.iterparse(io.StringIO(content))
            for _, el in it:
                _, _, el.tag = el.tag.rpartition('}')
            root_d = it.root
            for elem in root_d.iter():
                tag = elem.tag.lower()
                if tag in ('node','hazardmapnode','mapnode'):
                    lon = elem.get('lon') or elem.get('longitude')
                    lat = elem.get('lat') or elem.get('latitude')
                    iml = elem.get('iml') or elem.get('value')
                    if lon and lat and iml:
                        lons.append(float(lon)); lats.append(float(lat))
                        vals.append(float(iml))
            if lons:
                if debug: print(f'  XML parser: {len(lons)} nodes')
                return _apply_bbox(np.array(lons), np.array(lats),
                                   np.array(vals), bbox)
        except ET.ParseError:
            pass

    except Exception as e:
        if debug:
            print(f'  Exception: {e}')
    return None, None, None


def _is_float_row(line, sep):
    parts = line.split(sep)
    if len(parts) < 3:
        return False
    try:
        [float(p) for p in parts[:3]]
        return True
    except ValueError:
        return False


def _apply_bbox(lons, lats, vals, bbox):
    if bbox is None:
        return lons, lats, vals
    lon_min, lat_min, lon_max, lat_max = bbox
    mask = ((lons >= lon_min) & (lons <= lon_max) &
            (lats >= lat_min) & (lats <= lat_max))
    return lons[mask], lats[mask], vals[mask]


def fetch_map(model_id, imt, poe_prob, poe_years,
              soil='rock_vs30_800ms-1',
              agg_type='arithmetic', agg_level='0.5',
              bbox=None, debug=False):
    """
    Fetch a spatial hazard map through the SHARE /map cascade:
      Level 3 → get <hazardmaplocation> URL
      Level 4 → download and parse the actual grid data

    Returns dict with lons, lats, values arrays, or None.
    """
    print(f'Fetching map: {imt}, PoE={poe_prob}/{poe_years}yr')

    # Level 3: get the reference
    ref = get_map_reference(model_id, imt, poe_prob, poe_years,
                            agg_type, agg_level, soil, debug=debug)
    if ref is None:
        print('  ✗ Level 3 (reference) failed. Run Cell 11 to diagnose.')
        return None

    if debug:
        print(f'  Map reference received:')
        for k, v in ref.items():
            print(f'    {k}: {v[:80] if v else "None"}')

    # Level 4: download the actual data
    loc = ref.get('hazardmaplocation')
    if not loc:
        hmapid  = ref.get('hmapid',  '')
        hmapwms = ref.get('hmapwms', '')
        if hmapid or hmapwms:
            print(f'  No <hazardmaplocation> — probing data endpoints '
                  f'(hmapid={hmapid}, hmapwms={hmapwms})')
            # Candidates derived from hmapid (integer map ID)
            id_candidates = [
                f'{BASE_URL}/hazardmapdata?hmapid={hmapid}',
                f'{BASE_URL}/hazardmap?id={hmapid}',
                f'{BASE_URL}/mapdata?id={hmapid}',
                f'{BASE_URL}/hazardmap/{hmapid}',
                f'{BASE_URL}/map/data?id={hmapid}',
                f'{BASE_URL}/map/download?id={hmapid}',
                f'{BASE_URL}/hazardmap?hmapid={hmapid}',
                f'{BASE_URL}/hmapdata?id={hmapid}',
            ] if hmapid else []
            # Candidates derived from hmapwms layer name (e.g. "hmap1846")
            wms_candidates = [
                f'{BASE_URL}/{hmapwms}',
                f'{BASE_URL}/hazardmap?name={hmapwms}',
                f'{BASE_URL}/hazardmap?wmsname={hmapwms}',
                f'{BASE_URL}/mapdata?name={hmapwms}',
                (f'{BASE_URL}/wfs?service=WFS&version=2.0.0&request=GetFeature'
                 f'&typeName={hmapwms}&outputFormat=csv'),
                (f'{BASE_URL}/wcs?service=WCS&request=GetCoverage'
                 f'&coverageId={hmapwms}&format=text%2Fcsv'),
            ] if hmapwms else []
            for cand in id_candidates + wms_candidates:
                try:
                    r = requests.get(cand, timeout=20)
                    snippet = r.text[:80].replace("\n", " ") if r.text else ""
                    print(f'    HTTP {r.status_code}  {cand.split("/share/")[1][:55]}'
                          f'  → {snippet!r}')
                    if r.status_code == 200 and len(r.text) > 20:
                        _l, _a, _v = fetch_map_data_from_location(
                            cand, bbox=bbox, debug=debug)
                        if _l is not None and len(_l) > 0:
                            loc = cand
                            print(f'    ✓ Data found at: {cand}')
                            break
                except Exception as e:
                    print(f'    Error  {cand.split("/share/")[1][:55]}: {e}')
        if not loc:
            print('  ✗ No <hazardmaplocation> in reference response.')
            print(f'    Reference was: {ref}')
            print('    Run Cell 11 with debug=True for raw XML of Level 3.')
            return None

    lons, lats, vals = fetch_map_data_from_location(loc, bbox=bbox, debug=debug)

    if lons is None or len(lons) == 0:
        print(f'  ✗ Level 4 (data download) failed or returned 0 nodes.')
        print(f'    Location URL was: {loc}')
        print('    Run Cell 11 to inspect the data response.')
        return None

    print(f'  ✓ {len(lons)} nodes  |  '
          f'{imt} range: {vals.min():.4f}–{vals.max():.4f} g')
    return {'lons': lons, 'lats': lats, 'values': vals,
            'imt': imt, 'poe': poe_prob, 'n_nodes': len(lons),
            'location_url': loc,
            'wms_url': ref.get('hmapwms', '')}


def probe_map_raw(model_id, imt, poe_prob=None, poe_years=None,
                  agg_type=None, agg_level=None, soil=None, n_chars=800):
    """
    Print raw XML at each level of the /map cascade.
    Use in the debug cell to inspect exactly what the server returns.
    """
    levels = []
    # Level 1
    levels.append(f'{BASE_URL}/map?id={model_id}&imt={imt}')
    # Level 2 (if PoE given)
    if poe_prob:
        levels.append(f'{BASE_URL}/map?id={model_id}&imt={imt}'
                      f'&hmapexceedprob={poe_prob}&hmapexceedyears={poe_years or "1"}')
    # Level 3 (if all given)
    if poe_prob and agg_type and soil:
        levels.append(f'{BASE_URL}/map?id={model_id}&imt={imt}'
                      f'&hmapexceedprob={poe_prob}&hmapexceedyears={poe_years or "1"}'
                      f'&soiltype={soil}'
                      f'&aggregationtype={agg_type}&aggregationlevel={agg_level or "0.5"}')
    for lvl_n, url in enumerate(levels, 1):
        print(f'\n--- Level {lvl_n} ---')
        print(f'URL: {url}')
        try:
            r = requests.get(url, timeout=30)
            print(f'HTTP {r.status_code}  |  {len(r.text)} chars')
            print(r.text[:n_chars])
        except Exception as e:
            print(f'Error: {e}')


def resolve_poe(model_name, target_rp=475):
    """Return (prob, years) for the closest available RP from MAP_POES or config."""
    import math
    poes = MAP_POES.get(model_name, [])
    if not poes and config:
        for m in config['available_models']:
            if m['name'] == model_name:
                poes = m['parameters'].get('poes', [])
                break
    if not poes:
        return None, None
    best, best_diff = poes[0], float('inf')
    for p in poes:
        try:
            yr = float(p['years'])
            pr = float(p['prob'])
            ar = pr if yr == 1 else -math.log(1 - pr) / yr
            diff = abs(1 / ar - target_rp)
            if diff < best_diff:
                best_diff, best = diff, p
        except Exception:
            pass
    return best['prob'], best['years']


def load_config():
    try:
        with open(CONFIG_FILE) as f:
            cfg = yaml.safe_load(f)
        print(f'Config loaded — {cfg["location"]["latitude"]}N, '
              f'{cfg["location"]["longitude"]}E')
        return cfg
    except FileNotFoundError:
        print('ERROR: hazard_config.yaml not found. Run Notebook 1 first.')
        return None
    except Exception as e:
        print(f'ERROR: failed to load hazard_config.yaml: {e}')
        return None


print('API helpers loaded  (4-level /map cascade).')


---
## 4. Map plotting engine

Two plotting backends are provided:
- **Cartopy** (if installed): proper Plate Carée projection with Natural Earth coastlines and country borders
- **Fallback**: standard matplotlib axes with a contour-fill over an interpolated grid

Both backends use `scipy.interpolate.griddata` to resample the scattered API output onto a regular lat/lon grid before contouring.

In [ ]:
# ── colour map: white-yellow-orange-red-darkred (seismic hazard convention) ─
HAZARD_CMAP = 'YlOrRd'


def _make_grid(map_data, grid_res=0.1):
    """
    Resample scattered (lon, lat, value) triplets onto a regular grid.
    Returns (lon_grid, lat_grid, val_grid) as 2-D arrays.
    grid_res: desired grid spacing in degrees.
    """
    lons   = map_data['lons']
    lats   = map_data['lats']
    vals   = map_data['values']

    lon_min, lon_max = lons.min(), lons.max()
    lat_min, lat_max = lats.min(), lats.max()

    lon_g = np.arange(lon_min, lon_max + grid_res, grid_res)
    lat_g = np.arange(lat_min, lat_max + grid_res, grid_res)
    LON, LAT = np.meshgrid(lon_g, lat_g)

    VAL = griddata((lons, lats), vals, (LON, LAT), method='linear')
    return LON, LAT, VAL


def _colorbar_label(imt):
    return f'{imt} [g]'


def plot_map_cartopy(map_data, title='', vmin=None, vmax=None, ax=None,
                     log_scale=True, n_levels=15, show_colorbar=True):
    """Plot a hazard map using cartopy (proper geographic projection)."""
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    LON, LAT, VAL = _make_grid(map_data)
    masked = np.ma.masked_invalid(VAL)

    if vmin is None:
        vmin = np.nanpercentile(map_data['values'], 2)
    if vmax is None:
        vmax = np.nanpercentile(map_data['values'], 98)

    vmin = max(vmin, 1e-5)  # guard against log(0)

    if log_scale and vmin > 0:
        levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)
        norm   = LogNorm(vmin=vmin, vmax=vmax)
    else:
        levels = np.linspace(vmin, vmax, n_levels)
        norm   = None

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8),
                               subplot_kw={'projection': ccrs.PlateCarree()})

    ax.set_extent([LON.min(), LON.max(), LAT.min(), LAT.max()],
                  crs=ccrs.PlateCarree())

    cf = ax.contourf(LON, LAT, masked, levels=levels, cmap=HAZARD_CMAP,
                     norm=norm, transform=ccrs.PlateCarree(), extend='both')

    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, color='0.2')
    ax.add_feature(cfeature.BORDERS,   linewidth=0.5, color='0.4', linestyle=':')
    ax.add_feature(cfeature.LAND,      facecolor='none', edgecolor='0.5')
    ax.gridlines(draw_labels=True, linewidth=0.4, color='grey',
                 alpha=0.5, linestyle='--')
    ax.set_title(title, fontsize=11)

    if show_colorbar and own_fig:
        plt.colorbar(cf, ax=ax, label=_colorbar_label(map_data['imt']),
                     shrink=0.75, pad=0.08)
        plt.tight_layout()
        plt.show()

    return cf


def plot_map_simple(map_data, title='', vmin=None, vmax=None, ax=None,
                    log_scale=True, n_levels=15, show_colorbar=True):
    """Plot a hazard map using plain matplotlib (no cartopy required)."""
    LON, LAT, VAL = _make_grid(map_data)
    masked = np.ma.masked_invalid(VAL)

    if vmin is None:
        vmin = np.nanpercentile(map_data['values'], 2)
    if vmax is None:
        vmax = np.nanpercentile(map_data['values'], 98)

    vmin = max(vmin, 1e-5)  # avoid log(0)

    if log_scale and vmin > 0:
        levels = np.logspace(np.log10(vmin), np.log10(vmax), n_levels)
        norm   = LogNorm(vmin=vmin, vmax=vmax)
    else:
        levels = np.linspace(vmin, vmax, n_levels)
        norm   = None

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(10, 8))

    cf = ax.contourf(LON, LAT, masked, levels=levels, cmap=HAZARD_CMAP,
                     norm=norm, extend='both')
    ax.contour(LON, LAT, masked, levels=levels, colors='k',
               linewidths=0.2, alpha=0.3)

    ax.set_xlabel('Longitude [°E]', fontsize=10)
    ax.set_ylabel('Latitude [°N]',  fontsize=10)
    ax.set_aspect('equal')
    ax.grid(True, linewidth=0.4, alpha=0.4, linestyle='--')
    ax.set_title(title, fontsize=11)

    if show_colorbar and own_fig:
        plt.colorbar(cf, ax=ax, label=_colorbar_label(map_data['imt']),
                     shrink=0.75)
        plt.tight_layout()
        plt.show()

    return cf


def plot_map(map_data, title='', vmin=None, vmax=None, ax=None,
             log_scale=True, n_levels=15, show_colorbar=True):
    """Dispatcher: uses cartopy if available, otherwise plain matplotlib."""
    if HAS_CARTOPY:
        return plot_map_cartopy(map_data, title=title, vmin=vmin, vmax=vmax,
                                ax=ax, log_scale=log_scale, n_levels=n_levels,
                                show_colorbar=show_colorbar)
    else:
        return plot_map_simple(map_data, title=title, vmin=vmin, vmax=vmax,
                               ax=ax, log_scale=log_scale, n_levels=n_levels,
                               show_colorbar=show_colorbar)


def mark_cities(ax, cities, projection=None):
    """
    Overlay city markers on a map axis.
    cities: list of dicts with 'name', 'lat', 'lon'
    """
    transform = projection if projection else None
    for city in cities:
        kw = dict(transform=transform) if transform else {}
        ax.plot(city['lon'], city['lat'], 'k^', markersize=6,
                markerfacecolor='white', markeredgewidth=1.2, **kw)
        ax.text(city['lon'] + 0.1, city['lat'] + 0.1, city['name'],
                fontsize=8, color='black',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7),
                **kw)


print('Plotting engine loaded.')

---
## 5. Load configuration and define study region

In [ ]:
config = load_config()

if config:
    MODELS = {m['name']: m for m in config['available_models']
              if m['name'] and m['parameters'].get('imts')}
    print(f'Usable models: {list(MODELS.keys())}')

# ── Geographic bounding boxes ──────────────────────────────────────────────
REGIONS = {
    'Southern Italy + Sicily':  (11.0, 36.0, 17.5, 42.5),  # lon_min, lat_min, lon_max, lat_max
    'Italy':                    ( 6.5, 36.0, 19.0, 47.5),
    'Messina Strait (zoom)':    (14.5, 37.5, 16.5, 39.0),
    'Central Mediterranean':    ( 5.0, 33.0, 25.0, 48.0),
    'Europe (full model)':      None,  # no bbox = full API domain
}

# ── Reference cities for annotation ────────────────────────────────────────
CITIES = [
    {'name': 'Messina',    'lat': 38.19, 'lon': 15.55},
    {'name': 'Catania',    'lat': 37.50, 'lon': 15.09},
    {'name': "L'Aquila",   'lat': 42.35, 'lon': 13.40},
    {'name': 'Naples',     'lat': 40.85, 'lon': 14.27},
    {'name': 'Rome',       'lat': 41.90, 'lon': 12.50},
    {'name': 'Reggio Cal.','lat': 38.11, 'lon': 15.65},
    {'name': 'Palermo',    'lat': 38.12, 'lon': 13.36},
]

print('\nAvailable regions:')
for name, bbox in REGIONS.items():
    desc = f'bbox {bbox}' if bbox else 'full domain'
    print(f'  {name}: {desc}')

---
## 6. Probe the /map endpoint (run this before exercises)

This cell discovers which **exact PoE values and parameter names** the `/map` endpoint accepts for each model. The `/map` endpoint has historically used `hmapexceedprob`/`hmapexceedyears` as URL parameter names rather than `poe`/`timespanpoe`, and the exact numerical PoE values must match what the server has pre-computed.

Run this once after loading the config. The output will tell you which values to use in each exercise. If the probe returns nothing, use Cell 11 (debug) to inspect the raw XML.

In [ ]:
# ── Map parameter probe (walk all 4 cascade levels) ────────────────────────
# Run this after Cell 5 (load_config). It discovers valid PoEs, aggregations,
# and confirms the data location URL for each model.

if config:
    import math
    print('=== /map cascade probe ===\n')
    MAP_POES = {}

    for model in config['available_models']:
        if not model['name'] or not model['parameters'].get('imts'):
            continue
        mid  = model['id']
        name = model['name']
        soil = model['parameters']['soil_types'][0] if model['parameters'].get('soil_types') else 'rock_vs30_800ms-1'
        test_imt = 'PGA' if 'PGA' in model['parameters']['imts'] else model['parameters']['imts'][0]

        print(f'{"═"*60}')
        print(f'Model: {name}')

        # Level 1
        poes = probe_map_poes(mid, test_imt)
        MAP_POES[name] = poes
        if not poes:
            MAP_POES[name] = model['parameters'].get('poes', [])
            print(f'  Level 1 (PoEs): none from /map — using {len(MAP_POES[name])} from config')
        else:
            print(f'  Level 1 (PoEs): {len(poes)} found')
            for p in poes:
                ar = float(p["prob"]) if float(p["years"]) == 1 else (
                     -math.log(1 - float(p["prob"])) / float(p["years"]))
                rp = round(1 / ar)
                print(f'    prob={p["prob"]:>12s}  years={p["years"]}  → ~{rp}-yr RP')

        if not MAP_POES[name]:
            print('  No PoEs available. Skipping levels 2–4.')
            print()
            continue

        # Use the 475-yr RP PoE (or closest) for levels 2–4
        test_poe_prob, test_poe_years = resolve_poe(name, target_rp=475)
        if test_poe_prob is None:
            test_poe_prob, test_poe_years = MAP_POES[name][0]['prob'], MAP_POES[name][0]['years']

        # Level 2
        aggs = probe_map_aggregations(mid, test_imt, test_poe_prob, test_poe_years)
        if aggs:
            agg_labels = [a["type"] + " " + a["level"] for a in aggs]
            print(f'  Level 2 (aggregations): {agg_labels}')
        else:
            print(f'  Level 2 (aggregations): none found — using config values')
            aggs = model['parameters'].get('aggregations', [{'type':'arithmetic','level':'0.5'}])

        # Level 3: pick mean/arithmetic aggregation
        chosen_agg = next((a for a in aggs if a['type'] == 'arithmetic'), aggs[0])
        ref = get_map_reference(mid, test_imt, test_poe_prob, test_poe_years,
                                chosen_agg['type'], chosen_agg['level'], soil, debug=False)
        if ref:
            print(f'  Level 3 (reference):')
            for k, v in ref.items():
                print(f'    {k}: {(v[:90] + "…") if v and len(v) > 90 else v}')
        else:
            print(f'  Level 3 (reference): FAILED')
            print()
            continue

        # Level 4: test the data download
        loc = ref.get('hazardmaplocation')
        if loc:
            print(f'  Level 4 (data download):')
            try:
                r = requests.get(loc, timeout=30)
                print(f'    HTTP {r.status_code}  |  {len(r.text)} chars  |  '
                      f'first 100: {r.text[:100]!r}')
            except Exception as e:
                print(f'    Error: {e}')
        else:
            print(f'  Level 4: no <hazardmaplocation> in reference')
        print()

    print('Probe complete. MAP_POES populated.')
    print('If Level 4 shows data, exercises should work.')
    print('If Level 4 fails, run Cell 11 for detailed diagnostics.')

---
## 6. Exercise A: Basic PGA hazard map

**Task:** Download and plot the PGA hazard map for Southern Italy at the Eurocode design return period (475 years).

**Learning goals:**
- Identify the highest-hazard zones in the region
- Link hazard highs to tectonic structures (Calabrian Arc, Apennines, Etna)
- Read off the PGA at Messina and compare with the single-point result from Notebook 2

**Expected result:** PGA exceeds 0.4–0.5 g in the Messina Strait area, drops to 0.05–0.1 g in northern Sicily and coastal Calabria further from the main seismogenic sources.

In [ ]:
# ── Helper: resolve PoE from MAP_POES probe or config fallback ──────────
def resolve_poe(model_name, target_rp=475):
    """
    Return (poe_prob, poe_years) for the closest return period to target_rp.
    Reads from MAP_POES (populated by the probe cell) if available, else from config.
    """
    import math
    poes = MAP_POES.get(model_name, [])
    if not poes and config:
        # Fallback: read from hazard_config.yaml
        for m in config['available_models']:
            if m['name'] == model_name:
                poes = m['parameters'].get('poes', [])
                break
    if not poes:
        return None, None
    # Find closest to target return period
    best, best_diff = poes[0], float('inf')
    for p in poes:
        try:
            ar = float(p["prob"]) if float(p["years"]) == 1 else (
                 -math.log(1 - float(p["prob"])) / float(p["years"]))
            rp = 1 / ar if ar > 0 else 1e9
            if abs(rp - target_rp) < best_diff:
                best_diff = abs(rp - target_rp)
                best = p
        except Exception:
            pass
    return best['prob'], best['years']

# ============================================================
#  EXERCISE A — Basic PGA map at ~475-yr return period
#  Edit settings and run.
# ============================================================
EX_A = {
    'model_name' : 'European Seismic hazard Model 2020 (ESHM20)',
    'imt'        : 'PGA',
    'soil'       : 'rock_vs30_800ms-1',
    'agg_type'   : 'arithmetic',
    'agg_level'  : '0.5',
    'region'     : 'Southern Italy + Sicily',
    'log_scale'  : True,
    'debug'      : False,
}

# ── Auto-resolve closest 475-yr PoE from probed values ──────────────────
poe_prob, poe_years = resolve_poe(EX_A['model_name'], target_rp=475)
if poe_prob is None:
    # Manual fallback (ESHM20 annual rate for 475-yr RP)
    poe_prob, poe_years = '0.002103', '1'
    print(f'Using manual PoE fallback: {poe_prob} in {poe_years} yr')
else:
    print(f'Using probed PoE: prob={poe_prob}, years={poe_years}')

# --- Do not edit below this line ---
model = MODELS.get(EX_A['model_name'])
if model is None:
    print(f"Model not found. Options: {list(MODELS.keys())}")
else:
    map_a = fetch_map(
        model['id'], EX_A['imt'], poe_prob, poe_years,
        soil=EX_A['soil'],
        agg_type=EX_A['agg_type'], agg_level=EX_A['agg_level'],
        bbox=REGIONS[EX_A['region']],
        debug=EX_A['debug']
    )

    if map_a:
        print(f"  PGA range: {map_a['values'].min():.4f} – {map_a['values'].max():.4f} g")

        if HAS_CARTOPY:
            import cartopy.crs as ccrs
            fig, ax = plt.subplots(figsize=(10, 8),
                                   subplot_kw={'projection': ccrs.PlateCarree()})
            cf = plot_map_cartopy(
                map_a,
                title=f'PGA — ~475-yr RP — {EX_A["model_name"][:25]}… (mean)\n{EX_A["region"]}',
                ax=ax, log_scale=EX_A['log_scale'], show_colorbar=False)
            mark_cities(ax, CITIES, projection=ccrs.PlateCarree())
        else:
            fig, ax = plt.subplots(figsize=(10, 8))
            cf = plot_map_simple(
                map_a,
                title=f'PGA — ~475-yr RP — {EX_A["model_name"][:25]}… (mean)\n{EX_A["region"]}',
                ax=ax, log_scale=EX_A['log_scale'], show_colorbar=False)
            b = REGIONS[EX_A['region']]
            mark_cities(ax, [c for c in CITIES
                              if b[0] <= c['lon'] <= b[2] and b[1] <= c['lat'] <= b[3]])

        plt.colorbar(cf, ax=ax, label='PGA [g]', shrink=0.75, pad=0.08)
        plt.tight_layout()
        plt.show()
    else:
        print('\n→ fetch_map failed. Run Cell 11 (debug) to diagnose.')

---
## 7. Exercise B: Three return periods side by side

**Task:** Plot PGA maps for three return periods in a single figure:
- 101-yr RP (frequent shaking, ~39% in 50y)
- 475-yr RP (Eurocode design, 10% in 50y)
- 2500-yr RP (near-collapse, 2% in 50y)

**Learning goals:**
- Observe how the spatial pattern evolves with return period
- High-hazard zones are already apparent at short return periods, but low-hazard zones remain low — the *relative* pattern is stable
- Understand what it means to design at different safety levels

**Note:** All three maps use the same colour scale (`vmin` / `vmax`) so they are directly comparable.

In [ ]:
# ── Helper: resolve PoE from MAP_POES probe or config fallback ──────────
def resolve_poe(model_name, target_rp=475):
    """
    Return (poe_prob, poe_years) for the closest return period to target_rp.
    Reads from MAP_POES (populated by the probe cell) if available, else from config.
    """
    import math
    poes = MAP_POES.get(model_name, [])
    if not poes and config:
        # Fallback: read from hazard_config.yaml
        for m in config['available_models']:
            if m['name'] == model_name:
                poes = m['parameters'].get('poes', [])
                break
    if not poes:
        return None, None
    # Find closest to target return period
    best, best_diff = poes[0], float('inf')
    for p in poes:
        try:
            ar = float(p["prob"]) if float(p["years"]) == 1 else (
                 -math.log(1 - float(p["prob"])) / float(p["years"]))
            rp = 1 / ar if ar > 0 else 1e9
            if abs(rp - target_rp) < best_diff:
                best_diff = abs(rp - target_rp)
                best = p
        except Exception:
            pass
    return best['prob'], best['years']

# ============================================================
#  EXERCISE B — Three return periods, shared colour scale
# ============================================================
EX_B = {
    'model_name': 'European Seismic hazard Model 2020 (ESHM20)',
    'imt'       : 'PGA',
    'soil'      : 'rock_vs30_800ms-1',
    'agg_type'  : 'arithmetic',
    'agg_level' : '0.5',
    'region'    : 'Southern Italy + Sicily',
    'target_rps': [101, 475, 2500],
    'vmin': 0.01, 'vmax': 0.80,
}

# --- Do not edit below this line ---
model = MODELS.get(EX_B['model_name'])
if model is None:
    print(f"Model not found. Options: {list(MODELS.keys())}")
else:
    maps_b = []
    for rp in EX_B['target_rps']:
        poe_prob, poe_years = resolve_poe(EX_B['model_name'], target_rp=rp)
        if poe_prob is None:
            print(f'  No PoE found for ~{rp}-yr RP — skipping')
            continue
        import math
        ar = float(poe_prob) if float(poe_years) == 1 else (
             -math.log(1 - float(poe_prob)) / float(poe_years))
        actual_rp = round(1 / ar)
        print(f'  ~{rp}-yr RP → using prob={poe_prob}, years={poe_years} (~{actual_rp}-yr RP)')
        m = fetch_map(model['id'], EX_B['imt'], poe_prob, poe_years,
                      soil=EX_B['soil'],
                      agg_type=EX_B['agg_type'], agg_level=EX_B['agg_level'],
                      bbox=REGIONS[EX_B['region']])
        if m:
            m['label'] = f'~{actual_rp}-yr RP'
            maps_b.append(m)

    if maps_b:
        ncols = len(maps_b)
        if HAS_CARTOPY:
            import cartopy.crs as ccrs
            fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 7),
                                     subplot_kw={'projection': ccrs.PlateCarree()})
        else:
            fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 7))
        if ncols == 1: axes = [axes]

        cfs = []
        for ax, m in zip(axes, maps_b):
            cf = plot_map(m, title=m['label'],
                          vmin=EX_B['vmin'], vmax=EX_B['vmax'],
                          ax=ax, log_scale=True, show_colorbar=False)
            cfs.append(cf)
        fig.suptitle(f'PGA — {EX_B["model_name"][:30]}… (mean) — shared log scale',
                     fontsize=12)
        plt.colorbar(cfs[-1], ax=axes, label='PGA [g]', shrink=0.75)
        plt.tight_layout()
        plt.show()
    else:
        print('\n→ No maps fetched. Run Cell 11 (debug) to diagnose.')

---
## 8. Exercise C: Model difference map (ESHM20 − ESHM13)

**Task:** Compute and plot the difference in PGA at 475-yr RP between ESHM20 and ESHM13. Both maps must first be fetched and interpolated onto the same regular grid before differencing.

**Learning goals:**
- Quantify where the new model changed hazard estimates and by how much
- Identify regions of significant uplift (positive difference) and reduction
- Understand why model updates are important for engineering practice

**Colour scale:** Symmetric diverging palette (blue = ESHM20 lower, red = ESHM20 higher).

In [ ]:
# ── Helper: resolve PoE from MAP_POES probe or config fallback ──────────
def resolve_poe(model_name, target_rp=475):
    """
    Return (poe_prob, poe_years) for the closest return period to target_rp.
    Reads from MAP_POES (populated by the probe cell) if available, else from config.
    """
    import math
    poes = MAP_POES.get(model_name, [])
    if not poes and config:
        # Fallback: read from hazard_config.yaml
        for m in config['available_models']:
            if m['name'] == model_name:
                poes = m['parameters'].get('poes', [])
                break
    if not poes:
        return None, None
    # Find closest to target return period
    best, best_diff = poes[0], float('inf')
    for p in poes:
        try:
            ar = float(p["prob"]) if float(p["years"]) == 1 else (
                 -math.log(1 - float(p["prob"])) / float(p["years"]))
            rp = 1 / ar if ar > 0 else 1e9
            if abs(rp - target_rp) < best_diff:
                best_diff = abs(rp - target_rp)
                best = p
        except Exception:
            pass
    return best['prob'], best['years']

# ============================================================
#  EXERCISE C — ESHM20 minus ESHM13 difference map
# ============================================================
EX_C = {
    'imt'      : 'PGA',
    'soil'     : 'rock_vs30_800ms-1',
    'agg_type' : 'arithmetic',
    'agg_level': '0.5',
    'region'   : 'Southern Italy + Sicily',
    'target_rp': 475,
    'max_diff_g': 0.15,
}

# --- Do not edit below this line ---
model20 = MODELS.get('European Seismic hazard Model 2020 (ESHM20)')
model13 = MODELS.get('European Seismic Hazard Model 2013 (ESHM13)')

if model20 is None or model13 is None:
    print('Both ESHM13 and ESHM20 must be in the config. Re-run Notebook 1.')
else:
    poe20_prob, poe20_years = resolve_poe(model20['name'], target_rp=EX_C['target_rp'])
    poe13_prob, poe13_years = resolve_poe(model13['name'], target_rp=EX_C['target_rp'])

    if poe20_prob is None: poe20_prob, poe20_years = '0.002103', '1'
    if poe13_prob is None: poe13_prob, poe13_years = '0.1', '50'

    print(f'ESHM20 PoE: {poe20_prob} in {poe20_years} yr')
    print(f'ESHM13 PoE: {poe13_prob} in {poe13_years} yr')

    map20 = fetch_map(model20['id'], EX_C['imt'], poe20_prob, poe20_years,
                      soil=EX_C['soil'], agg_type=EX_C['agg_type'],
                      agg_level=EX_C['agg_level'], bbox=REGIONS[EX_C['region']])
    map13 = fetch_map(model13['id'], EX_C['imt'], poe13_prob, poe13_years,
                      soil=EX_C['soil'], agg_type=EX_C['agg_type'],
                      agg_level=EX_C['agg_level'], bbox=REGIONS[EX_C['region']])

    if map20 and map13:
        grid_res = 0.1
        lon_min = max(map20['lons'].min(), map13['lons'].min())
        lon_max = min(map20['lons'].max(), map13['lons'].max())
        lat_min = max(map20['lats'].min(), map13['lats'].min())
        lat_max = min(map20['lats'].max(), map13['lats'].max())
        lon_g = np.arange(lon_min, lon_max + grid_res, grid_res)
        lat_g = np.arange(lat_min, lat_max + grid_res, grid_res)
        LON, LAT = np.meshgrid(lon_g, lat_g)

        V20 = griddata((map20['lons'], map20['lats']), map20['values'],
                       (LON, LAT), method='linear')
        V13 = griddata((map13['lons'], map13['lats']), map13['values'],
                       (LON, LAT), method='linear')
        DIFF = np.ma.masked_invalid(V20 - V13)

        clim   = EX_C['max_diff_g']
        levels = np.linspace(-clim, clim, 21)

        if HAS_CARTOPY:
            import cartopy.crs as ccrs, cartopy.feature as cfeature
            fig, ax = plt.subplots(figsize=(10, 8),
                                   subplot_kw={'projection': ccrs.PlateCarree()})
            cf = ax.contourf(LON, LAT, DIFF, levels=levels, cmap='RdBu_r',
                             extend='both', transform=ccrs.PlateCarree())
            ax.add_feature(cfeature.COASTLINE, linewidth=0.7, color='0.2')
            ax.add_feature(cfeature.BORDERS,   linewidth=0.5, color='0.4', linestyle=':')
            ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4, linestyle='--')
        else:
            fig, ax = plt.subplots(figsize=(10, 8))
            cf = ax.contourf(LON, LAT, DIFF, levels=levels, cmap='RdBu_r', extend='both')
            ax.set_xlabel('Longitude [°E]'); ax.set_ylabel('Latitude [°N]')
            ax.set_aspect('equal')
            ax.grid(True, linewidth=0.4, alpha=0.4, linestyle='--')

        ax.contour(LON, LAT, DIFF, levels=[0], colors='k', linewidths=1.5)
        ax.set_title(f'ΔPGA = ESHM20 − ESHM13  [g]  |  ~{EX_C["target_rp"]}-yr RP  |  mean\n'
                     f'Red = ESHM20 higher, Blue = ESHM20 lower', fontsize=11)
        plt.colorbar(cf, ax=ax, label='ΔPGA [g]', shrink=0.75)
        plt.tight_layout(); plt.show()
        print(f'ΔPGA  mean={np.nanmean(DIFF):.4f}  p5={np.nanpercentile(DIFF,5):.4f}  '
              f'p95={np.nanpercentile(DIFF,95):.4f} g')
    else:
        print('\n→ One or both maps failed. Run Cell 11 (debug) to diagnose.')

---
## 9. Exercise D: Spectral ratio map SA[0.2s] / PGA

**Task:** Fetch PGA and SA[0.2s] maps at the same return period and compute their ratio. Plot the ratio map.

**Physical interpretation:** The ratio SA[0.2s]/PGA is a proxy for the *spectral amplification factor* at 0.2 s — how much more a short-period structure (T ≈ 0.2 s, roughly a 2-storey building) shakes compared to the ground itself. Where this ratio is high, the dominant seismic frequencies match the natural frequency of short-period structures. This reflects the frequency content of the ground motion, which is controlled by source-to-site distance, magnitude distribution, and crustal structure.

**Expected pattern:** The ratio tends to be higher close to large-magnitude source zones where near-field pulses dominate, and lower at sites receiving only high-frequency distant earthquakes.

In [ ]:
# ============================================================
#  EXERCISE D — Spectral ratio map SA[0.2s] / PGA
# ============================================================
EX_D = {
    'model_name' : 'European Seismic hazard Model 2020 (ESHM20)',
    'soil'       : 'rock_vs30_800ms-1',
    'agg_type'   : 'arithmetic',
    'agg_level'  : '0.5',
    'poe_prob'   : '0.002103',  # 475-yr RP annual rate
    'poe_years'  : '1',
    'region'     : 'Southern Italy + Sicily',
}

# --- Do not edit below this line ---
model = MODELS.get(EX_D['model_name'])
if model is None:
    print(f"Model not found. Options: {list(MODELS.keys())}")
else:
    print('Fetching PGA map...')
    map_pga = fetch_map(model['id'], 'PGA',
                        EX_D['poe_prob'], EX_D['poe_years'],
                        soil=EX_D['soil'],
                        agg_type=EX_D['agg_type'], agg_level=EX_D['agg_level'],
                        bbox=REGIONS[EX_D['region']])

    print('Fetching SA[0.20s] map...')
    map_sa02 = fetch_map(model['id'], 'SA[0.20s]',
                         EX_D['poe_prob'], EX_D['poe_years'],
                         soil=EX_D['soil'],
                         agg_type=EX_D['agg_type'], agg_level=EX_D['agg_level'],
                         bbox=REGIONS[EX_D['region']])

    if map_pga and map_sa02:
        # ── Regrid onto common grid ─────────────────────────────────────
        grid_res = 0.1
        lon_min = max(map_pga['lons'].min(), map_sa02['lons'].min())
        lon_max = min(map_pga['lons'].max(), map_sa02['lons'].max())
        lat_min = max(map_pga['lats'].min(), map_sa02['lats'].min())
        lat_max = min(map_pga['lats'].max(), map_sa02['lats'].max())

        lon_g = np.arange(lon_min, lon_max + grid_res, grid_res)
        lat_g = np.arange(lat_min, lat_max + grid_res, grid_res)
        LON, LAT = np.meshgrid(lon_g, lat_g)

        V_PGA  = griddata((map_pga['lons'],  map_pga['lats']),  map_pga['values'],
                          (LON, LAT), method='linear')
        V_SA02 = griddata((map_sa02['lons'], map_sa02['lats']), map_sa02['values'],
                          (LON, LAT), method='linear')

        with np.errstate(divide='ignore', invalid='ignore'):
            RATIO = np.where(V_PGA > 0, V_SA02 / V_PGA, np.nan)
        RATIO = np.ma.masked_invalid(RATIO)

        levels = np.linspace(0.5, 3.0, 21)

        if HAS_CARTOPY:
            import cartopy.crs as ccrs
            import cartopy.feature as cfeature
            fig, ax = plt.subplots(figsize=(10, 8),
                                   subplot_kw={'projection': ccrs.PlateCarree()})
            cf = ax.contourf(LON, LAT, RATIO, levels=levels, cmap='plasma',
                             extend='both', transform=ccrs.PlateCarree())
            ax.add_feature(cfeature.COASTLINE, linewidth=0.7, color='0.2')
            ax.add_feature(cfeature.BORDERS,   linewidth=0.5, color='0.4', linestyle=':')
            ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4, linestyle='--')
        else:
            fig, ax = plt.subplots(figsize=(10, 8))
            cf = ax.contourf(LON, LAT, RATIO, levels=levels, cmap='plasma', extend='both')
            ax.set_xlabel('Longitude [°E]')
            ax.set_ylabel('Latitude [°N]')
            ax.set_aspect('equal')
            ax.grid(True, linewidth=0.4, alpha=0.4, linestyle='--')

        ax.set_title('Spectral Ratio SA[0.2s] / PGA  |  475-yr RP  |  ESHM20 mean\n'
                     'High values → short-period structures face proportionally higher demand',
                     fontsize=11)
        plt.colorbar(cf, ax=ax, label='SA[0.2s] / PGA  [dimensionless]', shrink=0.75)
        plt.tight_layout()
        plt.show()

        print(f'Ratio range: {np.nanmin(RATIO):.2f} – {np.nanmax(RATIO):.2f}')
        # Query the already-computed RATIO grid at Messina coordinates
        _ratio_vals = np.array(RATIO.filled(np.nan)).ravel()
        _valid = ~np.isnan(_ratio_vals)
        messina_ratio = float(griddata(
            (LON.ravel()[_valid], LAT.ravel()[_valid]),
            _ratio_vals[_valid],
            [[15.55, 38.19]],
            method='nearest'
        ))
        print(f'Ratio at approximate Messina location: {messina_ratio:.2f}')

---
## 10. Exercise E: Interactive folium map

**Task:** Create an interactive HTML map where you can:
- Zoom in and out on the hazard grid
- Click on a city marker to see its PGA value
- Toggle between the hazard overlay and a basemap

**How to use:** After running this cell, a `.html` file is saved to disk. Open it in your browser for a fully interactive map.

**Learning goal:** Interactive maps are increasingly important for communicating seismic hazard to non-specialists (planners, engineers, civil protection agencies). This exercise shows how Python can produce web-ready outputs from the same hazard data.

In [ ]:
# ============================================================
#  EXERCISE E — Interactive folium web map
#  Requires: folium installed (Cell 1) and map_a from Exercise A
# ============================================================
FOLIUM_OUTPUT = 'hazard_map_interactive.html'

if not HAS_FOLIUM:
    print('folium not available. Install it with: pip install folium')
elif 'map_a' not in vars() or map_a is None:
    print('Run Exercise A first to fetch the map data.')
else:
    import folium
    from folium import plugins

    # ── Centre the map on the study region ────────────────────────────
    centre_lat = (map_a['lats'].min() + map_a['lats'].max()) / 2
    centre_lon = (map_a['lons'].min() + map_a['lons'].max()) / 2

    fmap = folium.Map(location=[centre_lat, centre_lon],
                      zoom_start=7,
                      tiles='CartoDB positron')

    # ── Build a colour-mapped circle marker for each grid node ────────
    # Normalise to [0,1] on log scale for colour mapping
    vals_log = np.log10(np.clip(map_a['values'], 1e-5, None))
    vmin_log = np.percentile(vals_log, 2)
    vmax_log = np.percentile(vals_log, 98)
    norm_vals = np.clip((vals_log - vmin_log) / (vmax_log - vmin_log), 0, 1)

    cmap_fn = plt.get_cmap('YlOrRd')

    def to_hex(v):
        r, g, b, _ = cmap_fn(float(v))
        return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

    # Only plot a representative subset if grid is very dense
    step = max(1, len(map_a['lons']) // 3000)
    print(f'Plotting {len(map_a["lons"][::step])} / {len(map_a["lons"])} grid nodes...')

    for lon, lat, val, nv in zip(map_a['lons'][::step],
                                  map_a['lats'][::step],
                                  map_a['values'][::step],
                                  norm_vals[::step]):
        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color=None,
            fill=True,
            fill_color=to_hex(nv),
            fill_opacity=0.7,
            tooltip=f'PGA = {val:.4f} g'
        ).add_to(fmap)

    # ── Add city markers ──────────────────────────────────────────────
    bbox = REGIONS['Southern Italy + Sicily']
    for city in CITIES:
        if bbox[0] <= city['lon'] <= bbox[2] and bbox[1] <= city['lat'] <= bbox[3]:
            # Find nearest grid node
            dists = np.hypot(map_a['lons'] - city['lon'], map_a['lats'] - city['lat'])
            nearest_idx = np.argmin(dists)
            pga_city = map_a['values'][nearest_idx]

            folium.Marker(
                location=[city['lat'], city['lon']],
                popup=folium.Popup(
                    f"<b>{city['name']}</b><br>"
                    f"PGA (475-yr, rock) = {pga_city:.4f} g<br>"
                    f"ESHM20 mean",
                    max_width=200
                ),
                tooltip=city['name'],
                icon=folium.Icon(color='darkblue', icon='home')
            ).add_to(fmap)

    # ── Add a simple legend ──────────────────────────────────────────
    legend_html = """
    <div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
                padding:10px;border:1px solid #ccc;border-radius:5px;font-size:12px">
    <b>PGA [g] — 475-yr RP</b><br>
    <i>ESHM20 mean — rock (Vs30 800 m/s)</i><br><br>
    <svg width="150" height="18"><defs>
      <linearGradient id="lg"><stop offset="0%" stop-color="#ffffb2"/>
      <stop offset="50%" stop-color="#fd8d3c"/><stop offset="100%" stop-color="#bd0026"/>
      </linearGradient></defs>
      <rect width="150" height="18" fill="url(#lg)"/></svg><br>
    Low &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; High
    </div>"""
    fmap.get_root().html.add_child(folium.Element(legend_html))

    fmap.save(FOLIUM_OUTPUT)
    print(f'Interactive map saved to: {FOLIUM_OUTPUT}')
    print('Open the .html file in a browser to explore the interactive map.')
    print('Click on city markers (home icons) to see PGA values.')

    # Optionally render inline in JupyterLab:
    try:
        from IPython.display import IFrame, display
        display(IFrame(FOLIUM_OUTPUT, width='100%', height='600px'))
    except Exception:
        print('(Could not embed inline — open the .html file in a browser)')

---
## 11. Debug: inspect raw map API response

In [ ]:
# ============================================================
#  CELL 11 — Detailed /map cascade diagnostics
#
#  Use this when fetch_map() fails. It walks all 4 levels
#  and prints full raw responses so you can see exactly
#  what the server returns at each step.
# ============================================================
DEBUG_MAP = {
    'model_id'  : '81',
    'imt'       : 'PGA',
    'soil'      : 'rock_vs30_800ms-1',
    'agg_type'  : 'arithmetic',
    'agg_level' : '0.5',
    'n_chars'   : 1200,
}

import math

# ── Levels 1–3 (raw XML) ────────────────────────────────────────────────────
poe_prob, poe_years = resolve_poe(
    next((m['name'] for m in config['available_models'] if m['id'] == DEBUG_MAP['model_id']), ''),
    target_rp=475)
if poe_prob is None:
    poe_prob, poe_years = '0.002103', '1'

print(f'Using PoE: {poe_prob} in {poe_years} yr\n')

probe_map_raw(
    DEBUG_MAP['model_id'], DEBUG_MAP['imt'],
    poe_prob=poe_prob, poe_years=poe_years,
    agg_type=DEBUG_MAP['agg_type'], agg_level=DEBUG_MAP['agg_level'],
    soil=DEBUG_MAP['soil'], n_chars=DEBUG_MAP['n_chars']
)

# ── Level 4: inspect the data URL ───────────────────────────────────────────
print('\n--- Level 4: data download ---')
ref = get_map_reference(
    DEBUG_MAP['model_id'], DEBUG_MAP['imt'],
    poe_prob, poe_years,
    DEBUG_MAP['agg_type'], DEBUG_MAP['agg_level'],
    DEBUG_MAP['soil'], debug=True)

if ref:
    loc = ref.get('hazardmaplocation')
    wms = ref.get('hmapwms')
    mid = ref.get('hmapid')
    print(f'\n  hmapid:           {mid}')
    print(f'  hmapwms:          {wms}')
    print(f'  hazardmaplocation:{loc}')

    if loc:
        print(f'\nFetching data from hazardmaplocation...')
        try:
            r = requests.get(loc, timeout=60)
            print(f'HTTP {r.status_code}  |  {len(r.text)} chars  |  '
                  f'Content-Type: {r.headers.get("Content-Type","?")}')
            print(f'\nFirst {DEBUG_MAP["n_chars"]} chars:')
            print(r.text[:DEBUG_MAP['n_chars']])
        except Exception as e:
            print(f'Error: {e}')

    if wms:
        print(f'\nWMS URL for reference (open in QGIS or browser):')
        print(f'  {wms}')
else:
    print('Level 3 (get_map_reference) returned None.')
    print('Check Levels 1–3 output above for the cause.')

---
## Summary of exercises

| Ex. | Cell | What was mapped | Key question |
|-----|------|-----------------|-------------|
| A | 6 | PGA — single RP — Southern Italy | Which structural zone has highest hazard? |
| B | 7 | PGA — three RPs, shared colour scale | How do spatial patterns shift with RP? |
| C | 8 | ESHM20 minus ESHM13 difference | Where did the new model change hazard most? |
| D | 9 | Spectral ratio SA[0.2s]/PGA | Where are short-period structures most at risk? |
| E | 10 | Interactive folium map with city markers | How does PGA vary between cities? |

---
### Further exploration

- **Change the region:** Edit `'region'` in any exercise to `'Italy'` or `'Central Mediterranean'` to zoom out.
- **Map SA at long periods:** Replace `'PGA'` with `'SA[1.00s]'` or `'SA[2.00s]'` in Exercise A and compare the spatial pattern — longer periods sample more distant sources.
- **Uncertainty map:** Fetch both `ordinal 0.16` and `ordinal 0.84` maps and compute the ratio 84th/16th as a spatial map of epistemic uncertainty. Where is the model most uncertain and why?
- **Site amplification overlay:** Multiply the rock PGA map by a simple Eurocode 8 site factor S for soft soil (e.g., S = 1.8 for class D) and compare the result with the rock map.

---
*EFEHR/SHARE API — http://appsrvr.share-eu.org:8080/share*  
*GeoINquire project — Messina Workshop, April 2026*